# Landlab demo: Perlin terrain, eroded

This notebook makes a game-style starting terrain from Perlin noise, then erodes it with
[Landlab](https://landlab.readthedocs.io/). Change the parameters and rerun to see what they do.

The scale is Minecraft-like: one grid cell is 1 m, and a 256 × 256 grid is 256 m across.

**How Landlab is organized.** Everything lives on a grid. Each grid point is a **node**, and each node
holds named **fields** such as `topographic__elevation`. **Components** are processes. Each component
reads some fields, updates others, and advances by one time step with `run_one_step(dt)`. You build
a model by choosing components and calling them in order.

**The model here.** Elevation $z$ changes by three processes:

$$
\frac{\partial z}{\partial t} = U \;-\; K A^m S^n \;+\; D \nabla^2 z
$$

- $U$ is **uplift**: rock pushed up everywhere. It is 0 by default, because we only want to erode a
  given terrain.
- $K A^m S^n$ is **river incision** (the stream power law). $A$ is drainage area, the area of land
  upstream of a point. $S$ is the local slope. Places with more upstream water and steeper slopes cut
  down faster. $K$ is erodibility. Component: `FastscapeEroder`.
- $D \nabla^2 z$ is **hillslope diffusion**: soil creeps downhill, which rounds off peaks and fills
  hollows. $D$ is diffusivity. Component: `LinearDiffuser`.

Drainage area comes from `FlowAccumulator`, which sends each node's water to its steepest downhill
neighbor and sums the area draining through each node.

The units are meters and years, but at this scale "years" mean little. Treat `K × total time` as how
strongly rivers cut, and `D × total time` as how much the terrain gets smoothed.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LightSource

from landlab import RasterModelGrid
from landlab.components import FastscapeEroder, FlowAccumulator, LinearDiffuser

from erosion_sim.render import render_terrain
from erosion_sim.terrain import fbm

## 1. Starting terrain

`fbm` (fractional Brownian motion) adds several layers ("octaves") of Perlin noise. The first layer
has features about `period` cells wide. Each following layer is `lacunarity` times finer and
`persistence` times weaker. The result is rescaled to [0, 1], and we multiply it by a height in
meters.

The plotting helper shows a **hillshade**: the terrain lit from the upper left, so slopes and
valleys are easy to see.

In [ ]:
def plot_hillshade(ax, z, title="", vmin=None, vmax=None, spacing=1.0, vert_exag=1.0):
    light = LightSource(azdeg=315, altdeg=45)
    shaded = light.shade(
        z, cmap=plt.cm.terrain, blend_mode="soft", vert_exag=vert_exag,
        dx=spacing, dy=spacing, vmin=vmin, vmax=vmax,
    )
    ax.imshow(shaded, origin="lower")
    ax.set_title(title)
    ax.set_axis_off()


def plot_field(ax, values, title="", cmap="viridis", label="", **kwargs):
    im = ax.imshow(values, origin="lower", cmap=cmap, **kwargs)
    plt.colorbar(im, ax=ax, label=label, shrink=0.8)
    ax.set_title(title)
    ax.set_axis_off()

In [ ]:
shape = (256, 256)
height = 64.0  # m

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), layout="constrained")
for ax, octaves in zip(axes, [1, 2, 3, 5]):
    plot_hillshade(ax, height * fbm(shape, seed=0, octaves=octaves), f"octaves={octaves}", 0, height)
plt.show()

## 2. The grid

`RasterModelGrid` is a regular square grid. Its outer edge nodes are **open boundaries**: their
elevation never changes, and water that reaches them leaves the model. Here the edges keep their
Perlin heights, so all water drains toward them. Only the interior **core nodes** change.

Landlab stores node fields as flat arrays, ordered row by row from the bottom. `reshape(grid.shape)`
turns one back into a 2D image.

In [ ]:
def make_grid(z0, spacing=1.0):
    # Return a raster grid whose elevation field is the 2D array z0.
    grid = RasterModelGrid(z0.shape, xy_spacing=spacing)
    grid.add_field("topographic__elevation", z0.ravel().copy(), at="node")
    return grid


grid = make_grid(height * fbm(shape, seed=0))
print("shape (rows, cols):", grid.shape, " spacing (m):", grid.dx)
print("nodes:", grid.number_of_nodes, " core nodes:", grid.number_of_core_nodes)
print("fields at nodes:", sorted(grid.at_node.keys()))

## 3. Run the model

Each time step does four things in order:

1. Raise every core node by `uplift * dt`.
2. Route water and compute drainage area (`FlowAccumulator`).
3. Cut down channels (`FastscapeEroder`).
4. Smooth hillslopes (`LinearDiffuser`).

**Pits** are closed hollows with no downhill neighbor. Without special handling, water flows into a
pit and stops. With `route_pits=True`, `DepressionFinderAndRouter` finds each pit and routes water
out through its lowest edge, as if it filled into a lake and overflowed. It is the slowest part of a
step; set `route_pits=False` to see the difference.

`FastscapeEroder` uses the FastScape algorithm (Braun & Willett 2013). It solves each step
implicitly, so large time steps stay stable.

In [ ]:
def erode(
    z0,
    spacing=1.0,
    uplift=0.0,         # m/yr, applied to core nodes only
    K=1e-3,             # erodibility
    m=0.4,              # drainage-area exponent
    n=1.0,              # slope exponent
    D=1e-3,             # hillslope diffusivity, m^2/yr
    dt=100.0,           # years per step
    steps=100,
    snapshot_every=10,
    route_pits=True,
    progress_every=0,   # print progress every this many steps; 0 turns it off
):
    # Erode the 2D starting surface z0. Return the grid, snapshot times, and 2D snapshots.
    grid = make_grid(z0, spacing)
    z = grid.at_node["topographic__elevation"]

    flow = FlowAccumulator(
        grid,
        flow_director="D8",
        depression_finder="DepressionFinderAndRouter" if route_pits else None,
    )
    incision = FastscapeEroder(grid, K_sp=K, m_sp=m, n_sp=n)
    diffusion = LinearDiffuser(grid, linear_diffusivity=D) if D > 0 else None

    times, snapshots = [0.0], [z.reshape(grid.shape).copy()]
    start = time.perf_counter()
    for step in range(1, steps + 1):
        z[grid.core_nodes] += uplift * dt
        flow.run_one_step()
        incision.run_one_step(dt)
        if diffusion is not None:
            diffusion.run_one_step(dt)
        if step % snapshot_every == 0:
            times.append(step * dt)
            snapshots.append(z.reshape(grid.shape).copy())
        if progress_every and step % progress_every == 0:
            elapsed = time.perf_counter() - start
            left = elapsed / step * (steps - step)
            print(f"step {step}/{steps}  {elapsed:.1f} s elapsed  ~{left:.1f} s left", flush=True)
    return grid, np.array(times), np.array(snapshots)

In [ ]:
z0 = height * fbm(shape, seed=0)

start = time.perf_counter()
grid, times, snapshots = erode(z0, progress_every=20)
print(f"{times[-1]:,.0f} yr in {time.perf_counter() - start:.1f} s")
print("fields at nodes after the run:", sorted(grid.at_node.keys()))

## 4. Look at the result

From left to right: the start, the end, how much each spot was lowered or raised, and drainage area.
Drainage area is on a log scale; the dark lines are the river network.

In [ ]:
z = snapshots[-1]
area = grid.at_node["drainage_area"].reshape(grid.shape)
change = z0 - z
limit = np.abs(change).max()

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), layout="constrained")
plot_hillshade(axes[0], z0, "start", 0, height)
plot_hillshade(axes[1], z, f"after {times[-1]:,.0f} yr", 0, height)
plot_field(axes[2], change, "lowered (red) / raised (blue)", cmap="RdBu_r", label="m", vmin=-limit, vmax=limit)
plot_field(axes[3], np.log10(np.maximum(area, grid.dx * grid.dy)), "drainage area", cmap="Blues", label="log10 area (m²)")
plt.show()

## 5. Watch it evolve

In [ ]:
picks = np.linspace(0, len(times) - 1, 6).astype(int)
fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 4), layout="constrained")
for ax, i in zip(axes, picks):
    plot_hillshade(ax, snapshots[i], f"{times[i]:,.0f} yr", 0, height)
plt.show()

## 6. 3D view

`render_terrain` (in `src/erosion_sim/render.py`) draws the heightmap as a lit 3D block with
PyVista. Color follows elevation. If you pass drainage area, rivers fade to blue. Useful options:

- `vert_exag`: stretch heights to make relief easier to see.
- `river_threshold`: log10 drainage area (m²) where blue starts. Lower it to show smaller streams.
- `azimuth`, `elevation`, `zoom`: the camera.
- `interactive=True`: open a separate window you can rotate with the mouse.

In [ ]:
images = [
    render_terrain(z0, vert_exag=1.5),
    render_terrain(z, area, vert_exag=1.5),
]
fig, axes = plt.subplots(1, 2, figsize=(18, 6), layout="constrained")
for ax, image, title in zip(axes, images, ["start", f"after {times[-1]:,.0f} yr"]):
    ax.imshow(image)
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

## 7. Rivers versus hillslopes

Large `K` cuts sharp valleys and, if pushed too far, wears everything down to the lowest edge.
Large `D` blurs the terrain. This cell runs five simulations, so it takes about a minute.

In [ ]:
cases = [
    dict(K=1e-4, D=1e-2),
    dict(K=1e-3, D=1e-2),
    dict(K=1e-3, D=1e-3),
    dict(K=1e-3, D=1e-4),
    dict(K=1e-2, D=1e-3),
]
fig, axes = plt.subplots(1, len(cases), figsize=(4 * len(cases), 4.3), layout="constrained")
for ax, case in zip(axes, cases):
    _, _, snaps = erode(z0, **case)
    plot_hillshade(ax, snaps[-1], f"K={case['K']:g}, D={case['D']:g}", 0, height)
plt.show()

## 8. Speed

Time for 50 steps at several grid sizes, with and without pit routing.

In [ ]:
for size in [64, 128, 256, 512]:
    z_start = height * fbm((size, size), seed=0, period=size / 2)
    row = f"{size:>4} × {size:<4}"
    for route_pits in (True, False):
        start = time.perf_counter()
        erode(z_start, steps=50, route_pits=route_pits)
        elapsed = time.perf_counter() - start
        row += f"   route_pits={route_pits!s:<5} {elapsed / 50 * 1000:7.1f} ms/step"
    print(row)

## 9. Other Landlab components to try

Landlab has many more components. Some that fit this setup, each a drop-in change to `erode`:

- `SpaceLargeScaleEroder`: replaces `FastscapeEroder`. Tracks bedrock and loose sediment separately,
  so rivers can deposit sediment as well as cut.
- `ErosionDeposition`: a simpler erosion-plus-deposition model with a single material.
- `TaylorNonLinearDiffuser`: replaces `LinearDiffuser`. Hillslope creep that speeds up sharply on
  steep slopes, which gives sharper ridges.
- `DepthDependentDiffuser` with `ExponentialWeatherer`: soil is produced from bedrock and creeps
  only where soil exists.
- `PrecipitationDistribution`: random storms, so runoff varies from step to step.
- `LakeMapperBarnes`: another way to handle pits, with options to fill them as lakes.
- `OverlandFlow`: water depth and flow during a single storm, at much shorter time scales.

The full list is in the [component docs](https://landlab.readthedocs.io/en/latest/user_guide/component_list.html).

## 10. Play

Change anything below and rerun. Some things to try:

- `seed`: a different terrain.
- `period`, `octaves`, `persistence`: the shape of the starting terrain.
- `K` and `D`: river cutting versus smoothing.
- `m` and `n`: the stream power exponents. `m/n` near 0.4–0.5 is typical.
- `steps`: how long erosion runs.
- `uplift`: push the interior up while it erodes, for mountain-building instead of pure wear.
- `route_pits`: turn off pit routing and see where water gets stuck.

In [ ]:
terrain = dict(shape=(256, 256), seed=1, period=128.0, octaves=5, persistence=0.5, lacunarity=2.0)
height = 64.0
params = dict(
    spacing=1.0,
    uplift=0.0,
    K=1e-3,
    m=0.4,
    n=1.0,
    D=1e-3,
    dt=100.0,
    steps=100,
    snapshot_every=10,
    route_pits=True,
    progress_every=20,
)

z0 = height * fbm(**terrain)
g, t, snaps = erode(z0, **params)
a = g.at_node["drainage_area"].reshape(g.shape)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
plot_hillshade(axes[0], z0, "start", 0, height)
plot_hillshade(axes[1], snaps[-1], f"after {t[-1]:,.0f} yr", 0, height)
plot_field(axes[2], np.log10(np.maximum(a, g.dx * g.dy)), "drainage area", cmap="Blues", label="log10 area (m²)")
plt.show()

fig, ax = plt.subplots(figsize=(12, 8), layout="constrained")
ax.imshow(render_terrain(snaps[-1], a, vert_exag=1.5))
ax.set_axis_off()
plt.show()